In [ ]:
!pip install langchain langchain-groq

In [ ]:
!pip install langchain-core

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=""
)

response = llm.invoke("What is langchain?")
print(response.content)

In [ ]:
# print(response)
# AIMessage(content="A vector database is...", response_metadata={...})

# print(response.content)      # the actual text
print(response.response_metadata)  # tokens used, model, etc.

In [ ]:
# from langchain_groq import ChatGroq        # Groq
# from langchain_openai import ChatOpenAI    # OpenAI
# from langchain_anthropic import ChatAnthropic  # Anthropic

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use past memory if available."),
    ("human", "{context} \nUser: {user_input}")
])

In [ ]:
formatted = prompt.format_messages(
    context="User likes coffee",
    user_input="What should I drink today?"
)

response = llm.invoke(formatted)
print(response.content)

In [ ]:
chain = prompt | llm

response = chain.invoke({
    "context": "User likes coffee",
    "user_input": "What should I drink today?"
})

print(response.content)

Since you’re a coffee fan, here are a few tasty options you can tailor to different times of day and moods:

### 1. **Morning Boost**
- **Classic Americano** – Espresso diluted with hot water. Light, smooth, and gives you that caffeine kick without the heaviness of a latte.
- **Cold Brew** – Steep coarsely ground beans in cold water for 12‑18 hours. It’s naturally less acidic, super refreshing, and can be served over ice with a splash of milk or a dash of vanilla syrup.

### 2. **Mid‑Morning Pick‑Me‑Up**
- **Flat White** – A velvety micro‑foam milk base with a double shot of espresso. It’s richer than a latte but still balanced.
- **Café Mocha** – Espresso + steamed milk + a drizzle of dark chocolate. Perfect if you want a hint of sweetness without reaching for a dessert.

### 3. **Afternoon Chill**
- **Iced Latte** – Cold milk, espresso, and ice. Add a dash of cinnamon or caramel for extra flavor.
- **Bulletproof Coffee** – Blend hot coffee with 1 Tbsp unsalted butter (or ghee) and 1 

In [6]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()


In [11]:
response = chain.invoke({
    "context": "explain in simpler terms",
    "user_input": "what is RAG in AIML?"
})

from IPython.display import display, Markdown

display(Markdown(response))


**RAG (Retrieval‑Augmented Generation) – a simple way to think about it**

| Piece | What it does | Everyday analogy |
|-------|--------------|-------------------|
| **Retriever** | Looks through a big “knowledge library” (documents, web pages, database entries) and pulls out the pieces that seem most relevant to your question. | Like a librarian who quickly finds a few books or articles that might answer your query. |
| **Generator** | Takes those retrieved snippets and uses a language model (e.g., GPT) to write a coherent answer, weaving the facts together. | Like a writer who reads the librarian‑handed pages and then crafts a clear, friendly response for you. |
| **Why combine them?** | Pure generation (just the language model) can hallucinate or miss up‑to‑date facts. Pure retrieval (just copy‑pasting) can be clunky and not flow naturally. RAG gives you the best of both: factual grounding + smooth, natural language. | Think of a student who both reads a textbook (retrieval) and then explains the concept in their own words (generation). |

### In plain language

1. **You ask a question.**  
2. The system first **searches** a huge collection of text (like searching Google or an internal document store) and grabs the most relevant paragraphs.  
3. Then a **language model** reads those paragraphs and **writes** an answer that’s easy to understand, citing the retrieved info when needed.  

### Quick example

- **Question:** “What’s the capital of Brazil?”  
- **Retriever:** Finds a snippet that says “Brazil’s capital is Brasília.”  
- **Generator:** Turns that into “The capital of Brazil is Brasília.”  

If the question were more complex, like “How does photosynthesis work?” the retriever would pull a few short explanations, and the generator would stitch them together into a clear, step‑by‑step answer.

### Key benefits

- **More accurate** – the answer is grounded in real documents.  
- **More up‑to‑date** – you can point the retriever at a fresh knowledge base, so the model can use the latest info without retraining.  
- **More flexible** – you can swap in different document sources (company manuals, scientific papers, news articles) depending on the task.

### Bottom line

RAG = **search + write**. It first **retrieves** relevant facts, then **generates** a natural‑language response that combines those facts into something easy for humans to read. Think of it as a smart assistant that both looks things up and explains them in its own words.

Why bother?
Because later you'll chain more steps after the LLM output. Those steps expect a plain string, not a message object.
prompt | llm | StrOutputParser() | next_step | ...
Run this and tell me output. Then we move to Retriever.

In [ ]:
!pip install langchain-community langchain-chroma

# RAG with Langchain

In [ ]:
import fitz

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

text = extract_text(".pdf")
print(text[:500])

Python Programming 
Programming is like giving a set of instructions to a computer to make it do what you want. 
Imagine you want to tell a robot how to make a sandwich. You’d break it down into steps, like "get 
the bread," "spread the butter," and "add Jam." 
In programming, you write these steps using a special language(here python) that the computer 
understands. When you write a program, you’re telling the computer exactly what to do, just like 
telling the robot how to make that sandwich! 


# RecursiveCharacterTextSplitter is smarter —

it tries to split on \n\n first, then \n, then spaces, then characters. So chunks break at natural boundaries like paragraphs instead of mid-sentence.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)
print("Total chunks:", len(chunks))
print("First chunk:\n", chunks[0])

Total chunks: 188
First chunk:
 Python Programming 
Programming is like giving a set of instructions to a computer to make it do what you want. 
Imagine you want to tell a robot how to make a sandwich. You’d break it down into steps, like "get 
the bread," "spread the butter," and "add Jam." 
In programming, you write these steps using a special language(here python) that the computer 
understands. When you write a program, you’re telling the computer exactly what to do, just like


In [ ]:
!pip install langchain-huggingface

In [1]:
# to delete old chunks
import chromadb

client = chromadb.PersistentClient(path="pdf_db")
client.delete_collection("langchain")

# ChromaDB via langchain

In [6]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2",
    model_kwargs={"device": "mps"} 
    # cache_folder=".cache/huggingface/hub"
)

vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory="pdf_db"
)

print("Stored:", vectorstore._collection.count(), "chunks")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6617.67it/s]


Stored: 188 chunks


In [ ]:
ls ~/.cache/huggingface/hub # run in terminal for cache hub

In [ ]:
cache_folder="~/.cache/huggingface/hub"

Before you manually encoded the query and called collection.query(). Now .as_retriever() handles all that. It returns Document objects instead of raw strings — so you access text via .page_content

# as_retriever

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

docs = retriever.invoke("why is python so popular")

for doc in docs:
    print(doc.page_content)
    print("---")

telling the robot how to make that sandwich! With programming, you can create games, websites, or 
even robots that follow your instructions. 
Python was created by Guido van Rossum in the late 1980s and released in 1991. It aimed to be an 
easy-to-read, high-level language. 
Why Python Is So Popular Now
1. Ease of Learning: Python's simple and readable syntax makes it accessible to beginners, 
fostering a large community of new developers.
---
fostering a large community of new developers. 
2. Versatile Applications: Python supports various domains, including web development, data 
analysis, artificial intelligence, scientific computing, automation, and more. 
3. Strong Libraries and Frameworks: The growth of libraries (like NumPy, Pandas, 
TensorFlow, Flask, and Django) has significantly enhanced Python's capabilities, making it 
suitable for diverse applications.
---
6. Cross-Platform Compatibility: Python runs on various platforms, allowing developers to 
create applications that w

type      what it does
similarity : plain cosine similarity, 
default mmr : picks diverse results, avoids repetitive chunks
similarity_score_threshold only returns chunks above a confidence score

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    api_key="",
    model="llama-3.3-70b-versatile"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions using only the context provided. decoate the answer with tables lines bullet points"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

chain = prompt | llm | StrOutputParser()

In [9]:
def ask(query):
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    return chain.invoke({"context": context, "question": query})


from IPython.display import display, Markdown

answer = ask("why is python so popular")
display(Markdown(answer))

**Why Python is So Popular**
=====================================

There are several reasons why Python is so popular. Here are some of the key reasons:

* **Ease of Learning**: Python's simple and readable syntax makes it accessible to beginners, fostering a large community of new developers.
* **Versatile Applications**: Python supports various domains, including:
  + Web development
  + Data analysis
  + Artificial intelligence
  + Scientific computing
  + Automation
  + More
* **Strong Libraries and Frameworks**: The growth of libraries like:
  | Library | Description |
  | --- | --- |
  | NumPy | Numerical computing |
  | Pandas | Data analysis |
  | TensorFlow | Machine learning |
  | Flask | Web development |
  | Django | Web development |
  has significantly enhanced Python's capabilities.
* **Community and Support**: A vibrant community contributes to continuous improvement, extensive documentation, and a wealth of online resources.
* **Rise of Data Science and AI**: The explosion of interest in data science, machine learning, and AI has propelled Python's popularity.
* **Cross-Platform Compatibility**: Python runs on various platforms, allowing developers to create applications that work across different environments seamlessly.
* **Corporate Adoption**: Many companies and tech giants, such as:
  + Google
  + Meta
  + Instagram
  have adopted Python, further validating its use in professional environments.

# HyDE RAG using Langchain

In [ ]:
import fitz

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

text = extract_text(".pdf")
print(text[:500])

Python Programming 
Programming is like giving a set of instructions to a computer to make it do what you want. 
Imagine you want to tell a robot how to make a sandwich. You’d break it down into steps, like "get 
the bread," "spread the butter," and "add Jam." 
In programming, you write these steps using a special language(here python) that the computer 
understands. When you write a program, you’re telling the computer exactly what to do, just like 
telling the robot how to make that sandwich! 


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)
print("Total chunks:", len(chunks))

Total chunks: 188


In [12]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "mps"}
)

vectorstore_hyde = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory="new_db"
)
print("Stored:", vectorstore_hyde._collection.count(), "chunks")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6037.58it/s]


Stored: 376 chunks


In [ ]:
# To delete old chunks
import chromadb

client = chromadb.PersistentClient(path="new_db")
client.delete_collection("langchain")

In [ ]:
retriever_hyde = vectorstore_hyde.as_retriever(search_kwargs={"k": 5})

docs = retriever_hyde.invoke("why is python so popular")

for doc in docs:
    print(doc.page_content)
    print("---")

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert. Generate a hypothetical document that would answer the question. Be detailed."),
    ("human", "{question}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

In [ ]:
question = "why is python so popular"
hypothetical_doc = hyde_chain.invoke({"question": question})
print(hypothetical_doc)

# Driver code

In [ ]:
def ask_hyde(query):
    # step 1 — generate hypothetical answer
    hypothetical_doc = hyde_chain.invoke({"question": query})
    
    # step 2 — retrieve using hypothetical answer instead of raw query
    docs = retriever_hyde.invoke(hypothetical_doc)
    
    # step 3 — answer using real retrieved chunks
    context = "\n\n".join([doc.page_content for doc in docs])
    return chain.invoke({"context": context, "question": query})

from IPython.display import display, Markdown
answer = ask_hyde("why is python so popular")
display(Markdown(answer))

# Hybrid RAG using Langchain 
Best Matching 25

In [ ]:
!pip install rank-bm25

#### BM25 + Vector hybrid


In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_texts(chunks)
bm25_retriever.k = 5

chroma_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)

In [ ]:
def ask_hybrid(query):
    docs = ensemble_retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    return chain.invoke({"context": context, "question": query})
from IPython.display import display, Markdown
answer = ask_hybrid("why is python so popular")
display(Markdown(answer))

BM25 finds exact keyword matches, Chroma finds semantic matches, EnsembleRetriever merges both with RRF automatically.